<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# Catalan Reasoning Dataset Preparation

Create a Catalan chain-of-thought reasoning dataset by translating Spanish examples.

## Pipeline

```
Multilingual-Thinking (Spanish) → aina-translator-es-ca → Catalan Reasoning Dataset
```

## Data Sources

| Source | License | Description |
|--------|---------|-------------|
| [HuggingFaceH4/Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) | Apache 2.0 | Spanish chain-of-thought reasoning examples |
| [projecte-aina/aina-translator-es-ca](https://huggingface.co/projecte-aina/aina-translator-es-ca) | Apache 2.0 | Spanish→Catalan translator (BSC) |

## Output

A Catalan reasoning dataset formatted for GPT-OSS Harmony fine-tuning.

## Setup

Install required dependencies.

In [ ]:
import subprocess
import sys

# Install required packages (datasets, tqdm from PyPI; ctranslate2, pyonmttok pre-installed in image)
packages = [
    "datasets",
    "huggingface_hub",
    "tqdm"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Verify AINA translator dependencies (pre-installed in tk-jupyter-fine-tuning image)
try:
    import ctranslate2
    import pyonmttok
    print("✅ Dependencies installed (ctranslate2 and pyonmttok from image)")
except ImportError as e:
    print(f"❌ Missing dependency: {e}")
    print("   Make sure you're using the 'Fine-Tuning Lab' Jupyter flavor")
    print("   which includes ctranslate2 and pyonmttok for the AINA translator")
    raise

In [1]:
import json
import os
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import snapshot_download
import ctranslate2
import pyonmttok
from tqdm import tqdm
from IPython.display import display, Markdown, HTML

# Helper functions for colored output
def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

# Output directory
OUTPUT_DIR = Path("./data")
OUTPUT_DIR.mkdir(exist_ok=True)

success("Libraries loaded")
info(f"Output directory: {OUTPUT_DIR.absolute()}")

---
## 1. Load Multilingual-Thinking Dataset

Download and filter Spanish examples from the dataset.

In [2]:
# Load the dataset
info("Loading Multilingual-Thinking dataset from HuggingFace...")
dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")

success(f"Loaded {len(dataset)} examples")

# Show language distribution
from collections import Counter
lang_counts = Counter(dataset["reasoning_language"])
print("\nLanguage distribution:")
for lang, count in lang_counts.most_common():
    print(f"  {lang}: {count} examples")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]


Language distribution:
  French: 200 examples
  English: 200 examples
  German: 200 examples
  Spanish: 200 examples
  Italian: 200 examples


In [3]:
# Filter Spanish examples
spanish_examples = dataset.filter(lambda x: x["reasoning_language"] == "Spanish")

success(f"Found {len(spanish_examples)} Spanish examples")

# Show a sample
print("\n" + "="*60)
print("SAMPLE SPANISH EXAMPLE")
print("="*60)
sample = spanish_examples[0]
print(f"\nUser: {sample['user'][:200]}...")
print(f"\nAnalysis (thinking): {sample['analysis'][:300]}...")
print(f"\nFinal response: {sample['final'][:200]}...")

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]


SAMPLE SPANISH EXAMPLE

User: I'd like to plan a trip to Rome for 7 days. I want to see the main attractions like the Colosseum, Vatican City, and the Sistine Chapel, but I also want to explore some lesser-known sites. I'm a foodi...

Analysis (thinking): Perfecto, veamos. El usuario quiere un viaje de 7 días a Roma con un equilibrio entre las atracciones principales, algunos lugares ocultos, buena comida y momentos de relajación. Mencionó el Coliseo, el Vaticano, la Capilla Sixtina y quiere explorar lugares menos conocidos. Además, es un amante de l...

Final response: **Rome 7-Day Itinerary: History, Food, & Relaxation**  
Hi little explorer! Let’s plan a fun and tasty trip to Rome where you’ll see cool places, eat amazing food, and take breaks to relax. Here’s you...


---
## 2. Load AINA Translator

Download and initialize the Spanish→Catalan translator from Barcelona Supercomputing Center.

In [4]:
# Download translator model
info("Downloading aina-translator-es-ca from HuggingFace...")
translator_dir = snapshot_download(
    repo_id="projecte-aina/aina-translator-es-ca",
    revision="main"
)

success(f"Translator downloaded to: {translator_dir}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

shared_vocabulary.txt: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/1.43M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

In [5]:
# Initialize tokenizer and translator
info("Initializing tokenizer and translator...")

tokenizer = pyonmttok.Tokenizer(
    mode="none",
    sp_model_path=os.path.join(translator_dir, "spm.model")
)

# Check CUDA availability
cuda_available = ctranslate2.get_cuda_device_count() > 0
device = "cuda" if cuda_available else "cpu"

# Initialize translator with explicit compute type for GPU optimization
translator = ctranslate2.Translator(
    translator_dir,
    device=device,
    compute_type="float16" if cuda_available else "default",  # FP16 on GPU for speed
    inter_threads=1,
    intra_threads=4 if device == "cpu" else 1
)

# Verify actual device being used
print(f"CUDA devices available: {ctranslate2.get_cuda_device_count()}")
print(f"Translator device: {device}")
print(f"Compute type: {'float16' if cuda_available else 'default'}")

success(f"Translator initialized on {device.upper()}")

CUDA devices available: 1
Translator device: cuda
Compute type: float16


In [6]:
# Test the translator
test_text = "Bienvenido al proyecto de razonamiento matemático."
tokenized = tokenizer.tokenize(test_text)
translated = translator.translate_batch([tokenized[0]])
result = tokenizer.detokenize(translated[0].hypotheses[0])

print("Translation test:")
print(f"  Spanish: {test_text}")
print(f"  Catalan: {result}")
success("Translator working!")

Translation test:
  Spanish: Bienvenido al proyecto de razonamiento matemático.
  Catalan: Benvingut al projecte de raonament matemàtic.


---
## 3. Translation Function

Define a function to translate text while preserving structure.

In [7]:
def translate_es_to_ca(text: str) -> str:
    """
    Translate Spanish text to Catalan.
    
    Uses batched translation for efficiency.
    """
    if not text or not text.strip():
        return text
    
    # Split into sentences for better translation of long texts
    sentences = []
    current = ""
    
    for char in text:
        current += char
        if char in '.!?\n' and len(current) > 10:
            sentences.append(current.strip())
            current = ""
    
    if current.strip():
        sentences.append(current.strip())
    
    if not sentences:
        return text
    
    # Tokenize all sentences
    tokenized_batch = [tokenizer.tokenize(sent)[0] for sent in sentences if sent]
    
    # Translate entire batch at once (much faster!)
    try:
        results = translator.translate_batch(
            tokenized_batch,
            max_batch_size=32,  # Process up to 32 sentences at once
            beam_size=2,        # Faster with smaller beam
            max_decoding_length=256
        )
        
        translated_sentences = [
            tokenizer.detokenize(result.hypotheses[0]) 
            for result in results
        ]
    except Exception as e:
        # Fallback: return original on error
        return text
    
    return " ".join(translated_sentences)


# Test with longer text
test_long = """Para resolver este problema matemático, primero necesitamos identificar las variables. 
Después, aplicaremos las fórmulas correspondientes. 
Finalmente, verificaremos el resultado."""

translated_long = translate_es_to_ca(test_long)
print("Long text translation test:")
print(f"\nSpanish:\n{test_long}")
print(f"\nCatalan:\n{translated_long}")

Long text translation test:

Spanish:
Para resolver este problema matemático, primero necesitamos identificar las variables. 
Después, aplicaremos las fórmulas correspondientes. 
Finalmente, verificaremos el resultado.

Catalan:
Per resoldre aquest problema matemàtic, primer necessitem identificar les variables. Després, aplicarem les fórmules corresponents. Finalment, verificarem el resultat.


---
## 4. Translate Spanish Examples to Catalan

Translate all Spanish reasoning examples to Catalan.

In [8]:
def translate_example(example: dict) -> dict:
    """
    Translate a single example from Spanish to Catalan.
    
    Translates:
    - user: The user's question/prompt
    - analysis: The chain-of-thought reasoning
    - final: The final response
    
    Keeps:
    - developer: System message (usually in English, keep as-is)
    """
    return {
        "reasoning_language": "Catalan",
        "developer": example["developer"],  # Keep system message
        "user": translate_es_to_ca(example["user"]),
        "analysis": translate_es_to_ca(example["analysis"]),
        "final": translate_es_to_ca(example["final"]),
        "original_language": "Spanish",  # Track source
    }


# Translate all Spanish examples
info(f"Translating {len(spanish_examples)} Spanish examples to Catalan...")
info("This may take a few minutes...")

catalan_examples = []
errors = []

for i, example in enumerate(tqdm(spanish_examples, desc="Translating")):
    try:
        translated = translate_example(example)
        catalan_examples.append(translated)
    except Exception as e:
        errors.append({"index": i, "error": str(e)})

success(f"Translated {len(catalan_examples)} examples to Catalan")
if errors:
    error(f"Failed to translate {len(errors)} examples")

Translating: 100%|██████████| 200/200 [01:22<00:00,  2.42it/s]


In [9]:
# Show a translated example
print("="*60)
print("SAMPLE CATALAN TRANSLATION")
print("="*60)

if catalan_examples:
    sample_ca = catalan_examples[0]
    sample_es = spanish_examples[0]
    
    print("\n--- USER MESSAGE ---")
    print(f"Spanish: {sample_es['user'][:150]}...")
    print(f"Catalan: {sample_ca['user'][:150]}...")
    
    print("\n--- CHAIN-OF-THOUGHT (analysis) ---")
    print(f"Spanish: {sample_es['analysis'][:200]}...")
    print(f"Catalan: {sample_ca['analysis'][:200]}...")
    
    print("\n--- FINAL RESPONSE ---")
    print(f"Spanish: {sample_es['final'][:150]}...")
    print(f"Catalan: {sample_ca['final'][:150]}...")

SAMPLE CATALAN TRANSLATION

--- USER MESSAGE ---
Spanish: I'd like to plan a trip to Rome for 7 days. I want to see the main attractions like the Colosseum, Vatican City, and the Sistine Chapel, but I also wa...
Catalan: I'd like to pla a trip to Rome for 7 days. I want to see the main attractions like the Colosseum, Vatican City, and the Sistine Chapel, but I also wan...

--- CHAIN-OF-THOUGHT (analysis) ---
Spanish: Perfecto, veamos. El usuario quiere un viaje de 7 días a Roma con un equilibrio entre las atracciones principales, algunos lugares ocultos, buena comida y momentos de relajación. Mencionó el Coliseo, ...
Catalan: Perfecte, a veure. L'usuari vol un viatge de 7 dies a Roma amb un equilibri entre les atraccions principals, alguns llocs ocults, bon menjar i moments de relaxació. Va esmentar el Coliseu, el Vaticà, ...

--- FINAL RESPONSE ---
Spanish: **Rome 7-Day Itinerary: History, Food, & Relaxation**  
Hi little explorer! Let’s plan a fun and tasty trip to Rome where you’ll 

---
## 5. Format for GPT-OSS Harmony Fine-tuning

Convert to the Harmony format expected by GPT-OSS.

In [10]:
def format_for_harmony(example: dict) -> dict:
    """
    Format example for GPT-OSS Harmony fine-tuning.
    
    Harmony format expects:
    - messages: list of {role, content}
    - Assistant message has 'thinking' field for chain-of-thought
    """
    # System message includes reasoning language instruction
    system_content = f"""reasoning language: Catalan

{example['developer']}"""
    
    messages = [
        {
            "role": "system",
            "content": system_content
        },
        {
            "role": "user",
            "content": example["user"]
        },
        {
            "role": "assistant",
            "content": example["final"],
            "thinking": example["analysis"]  # Chain-of-thought
        }
    ]
    
    return {
        "messages": messages,
        "language": "Catalan",
        "original_language": example.get("original_language", "Spanish")
    }


# Format all examples
info("Formatting examples for Harmony fine-tuning...")

harmony_examples = [format_for_harmony(ex) for ex in catalan_examples]

success(f"Formatted {len(harmony_examples)} examples for Harmony")

In [11]:
# Show formatted example
print("="*60)
print("HARMONY FORMAT EXAMPLE")
print("="*60)

if harmony_examples:
    sample = harmony_examples[0]
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:1500] + "...")

HARMONY FORMAT EXAMPLE
{
  "messages": [
    {
      "role": "system",
      "content": "reasoning language: Catalan\n\nYou are an AI that formats its responses in simple, easy to understand language for children"
    },
    {
      "role": "user",
      "content": "I'd like to pla a trip to Rome for 7 days. I want to see the main attractions like the Colosseum, Vatican City, and the Sistine Chapel, but I also want to explori some lesser-known sites. I'm a foodie who loves Italian cuisine, so recommend some must-visit restaurants and food markets. També, I'd like some down time to relax. Can you help em create an itinerary that balanços sightseeing, culinary adventures, and relaxion?"
    },
    {
      "role": "assistant",
      "content": "**Rome 7-Day Itinerary: Història, Alimentació i Relaxació** Hi little explorer! Let's plan a fun and tasty trip to Rome where you'll see cool places, eat amazing food, and take breaks to relax. Heus aquí el vostre pla: --- **Day 1: Ancient Rome & P

---
## 6. Save Dataset

Save the Catalan reasoning dataset for fine-tuning.

In [12]:
# Save as JSONL (one JSON object per line)
output_file = OUTPUT_DIR / "catalan_reasoning_train.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for example in harmony_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

success(f"Saved {len(harmony_examples)} examples to {output_file}")

# Also save metadata
metadata = {
    "dataset_name": "catalan_reasoning",
    "source_dataset": "HuggingFaceH4/Multilingual-Thinking",
    "source_language": "Spanish",
    "target_language": "Catalan",
    "translator": "projecte-aina/aina-translator-es-ca",
    "num_examples": len(harmony_examples),
    "format": "Harmony (GPT-OSS)",
    "license": "Apache 2.0",
    "created_by": "Thinkube AI Lab"
}

metadata_file = OUTPUT_DIR / "metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

success(f"Saved metadata to {metadata_file}")

In [13]:
# Verify the saved file
info("Verifying saved dataset...")

with open(output_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"\nFile: {output_file}")
print(f"Lines: {len(lines)}")
print(f"Size: {output_file.stat().st_size / 1024:.1f} KB")

# Parse first and last to verify
first = json.loads(lines[0])
last = json.loads(lines[-1])

print(f"\nFirst example language: {first.get('language')}")
print(f"Last example language: {last.get('language')}")

success("Dataset verified!")


File: data/catalan_reasoning_train.jsonl
Lines: 200
Size: 807.4 KB

First example language: Catalan
Last example language: Catalan


---
## Summary

Dataset preparation complete!

In [14]:
print("="*60)
print("DATASET PREPARATION COMPLETE")
print("="*60)
print(f"\nSource: HuggingFaceH4/Multilingual-Thinking (Spanish)")
print(f"Translator: projecte-aina/aina-translator-es-ca")
print(f"Output: {output_file}")
print(f"Examples: {len(harmony_examples)}")
print(f"Format: Harmony (GPT-OSS compatible)")
print(f"License: Apache 2.0")
print("\n" + "="*60)
print("\nNext step: Run 02-finetune-gpt-oss.ipynb to fine-tune GPT-OSS")

DATASET PREPARATION COMPLETE

Source: HuggingFaceH4/Multilingual-Thinking (Spanish)
Translator: projecte-aina/aina-translator-es-ca
Output: data/catalan_reasoning_train.jsonl
Examples: 200
Format: Harmony (GPT-OSS compatible)
License: Apache 2.0


Next step: Run 02-finetune-gpt-oss.ipynb to fine-tune GPT-OSS


---
## Next Steps

1. **Notebook 02**: Fine-tune GPT-OSS on this Catalan reasoning dataset
2. **Notebook 03**: Evaluate on ALIA-math-test benchmark

---

*Dataset created using open-source tools from Barcelona Supercomputing Center (Projecte AINA).*